# ⛑️ Safety Helmet Detection — Custom YOLOv8/YOLOv11 Training on Google Colab

Train a high-accuracy (95%+) custom YOLO model for **Safety Helmet & Head Compliance Detection**.

> **Important**: Enable GPU in Google Colab before running:
> `Runtime` ➔ `Change runtime type` ➔ select **T4 GPU** (or A100/V100 if available).

In [ ]:
# Step 1: Check GPU & Install Required Libraries
!nvidia-smi
!pip install -q ultralytics roboflow

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

## 📥 Step 2: Download Dataset
Choose **Option A (Roboflow Export Code)** OR **Option B (Upload Dataset Zip)**.

In [ ]:
# =========================================================================
# OPTION A: Roboflow Direct Download (Paste your snippet from Roboflow Export)
# =========================================================================
from roboflow import Roboflow

# Replace with your API key & project details if using Roboflow:
ROBOFLOW_API_KEY = "AydYTkTEwRNm3fM0n8yl"  # <-- Replace with your API key
WORKSPACE_NAME = "YOUR_WORKSPACE"        # <-- Replace with workspace name
PROJECT_NAME = "safety-helmet-dataset"   # <-- Replace with project name
VERSION = 1

try:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
    version = project.version(VERSION)
    dataset = version.download("yolov8")
    data_yaml_path = f"{dataset.location}/data.yaml"
    print(f"Dataset downloaded successfully to: {dataset.location}")
except Exception as e:
    print(f"Option A not configured or failed ({e}). Use Option B if uploading zip manually.")

In [ ]:
# =========================================================================
# OPTION B: Upload Dataset Zip File (If you downloaded zip from Roboflow)
# =========================================================================
import os, glob, zipfile
from google.colab import files

# Only run if Option A was not used
if 'data_yaml_path' not in locals() or not os.path.exists(data_yaml_path):
    print("Upload your dataset .zip file from your computer:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/dataset')
            print(f"Extracted {filename} to /content/dataset")

    found_yamls = glob.glob('/content/dataset/**/data.yaml', recursive=True) + glob.glob('/content/**/data.yaml', recursive=True)
    if found_yamls:
        data_yaml_path = found_yamls[0]
        print(f"Found data.yaml at: {data_yaml_path}")

In [ ]:
# Verify and inspect dataset classes
import yaml

with open(data_yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

print("="*40)
print("DATASET CONFIGURATION:")
print("Classes:", data_cfg.get('names'))
print("Num classes:", data_cfg.get('nc'))
print("="*40)

## 🚀 Step 3: Train YOLOv8 / YOLOv11 Model

For **95%+ accuracy**, we recommend `yolov8m.pt` or `yolo11m.pt` (Medium model) which captures small distant helmets significantly better than nano models.

In [ ]:
from ultralytics import YOLO

# Choose base model: 'yolov8m.pt' (Medium) or 'yolo11m.pt' for highest accuracy on distant/small objects
# Use 'yolov8s.pt' if you want faster inference.
base_model = 'yolov8m.pt'
model = YOLO(base_model)

# Train with optimal hyper-parameters for small helmet objects
results = model.train(
    data=data_yaml_path,
    epochs=100,           # 80-120 epochs is ideal
    imgsz=640,            # Can increase to 1024 if GPU memory allows for small objects
    batch=16,
    name='helmet_detector_v8m',
    patience=20,          # Early stopping if no improvement for 20 epochs
    augment=True,         # Data augmentation
    mosaic=1.0,           # Helps detect small objects
    mixup=0.10,
    fliplr=0.5,           # Left-right flip
    flipud=0.0,
    degrees=10.0,         # Slight rotations (head tilting)
    scale=0.5,            # Scale jitter for different camera distances
    hsv_h=0.015,          # Color variations (different lighting/helmet shades)
    hsv_s=0.7,
    hsv_v=0.4,
    save=True,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu'
)

## 📊 Step 4: Validate Model & Check Accuracy (mAP)

In [ ]:
# Run validation on test/val set
metrics = model.val()

print("="*40)
print(f"mAP @ 0.50:       {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
print(f"mAP @ 0.50-0.95:  {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
print(f"Precision:        {metrics.box.mp:.4f} ({metrics.box.mp*100:.1f}%)")
print(f"Recall:           {metrics.box.mr:.4f} ({metrics.box.mr*100:.1f}%)")
print("="*40)

In [ ]:
# Display results and confusion matrix
from IPython.display import Image, display
import glob

plot_files = glob.glob('runs/detect/helmet_detector_v8m*/results.png') + \
             glob.glob('runs/detect/helmet_detector_v8m*/confusion_matrix.png') + \
             glob.glob('runs/detect/helmet_detector_v8m*/val_batch0_pred.jpg')

for p in plot_files:
    print(f"Plot: {p}")
    display(Image(p))

## 💾 Step 5: Download `best.pt` Weights File

Download `best.pt` and place it in your local project folder: `apps/helmet_detection/best.pt`.

In [ ]:
import glob
from google.colab import files

# Locate best.pt
best_weights = glob.glob('runs/detect/helmet_detector_v8m*/weights/best.pt')
if best_weights:
    latest_best = best_weights[-1]
    print(f"Downloading {latest_best} to your local machine...")
    files.download(latest_best)
else:
    print("Error: best.pt weights not found. Check runs/detect directory.")